# Modelo de Classificação: Cats vs Dogs (imagens 224 x 224)

* Esse trabalho busca classificar imagens de gatos e cachorros através de uma CNN, inspirada na AlexNet.
* As imagens foram obtidas através de uma base de dados do kaggle com aproximadamente 25mil dados, que foram utilizados para treinar e testar o modelo posteriormente.
* Trabalho realizado para o picme de ``Processamento de Imagens Cardíacas`` com o intuito de praticar a teoria estudada até o momento atual e motivar uma competição entre os estudantes envolvidos.

**Diferenças em relação ao `cats_and_dogs.ipynb`:**
* As imagens são padronizadas em **224 x 224**, mantendo a proporção: o lado menor vira 224 e o excesso do lado maior é cortado no centro (antes, todas eram esticadas para 100 x 94).
* A divisão entre treino, validação e teste é feita **antes** do data augmentation, e o augmentation é aplicado **somente nas imagens de treino**. Validação e teste ficam só com imagens originais.
* As imagens ficam guardadas como `uint8` (0 a 255) e só viram `float` (0 a 1) por lote, para caber na memória.
* O treino tem menos épocas (a constante `EPOCAS`, na seção 6) e o histórico do treino é gravado pelo próprio notebook.
* O resto (arquitetura, otimizador, taxa de aprendizado, dropout, batch size, seed) continua igual.

## 1. Configuraçaõ Inicial

In [ ]:
%pip install kagglehub torch numpy pillow scipy matplotlib

In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, Subset, TensorDataset

SEED = 18
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

In [ ]:
import kagglehub

caminho = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print(caminho)

## 2. Exploração dos Dados

In [ ]:
for pasta, subpastas, arquivos in os.walk(caminho):
    print(pasta, "-", len(arquivos), "arquivos")

In [ ]:
info = []          # um item por arquivo que abriu como imagem
nao_abriram = []   # arquivos que não abriram como imagem

for pasta, _, nomes in os.walk(caminho):
    for nome in nomes:
        arquivo = os.path.join(pasta, nome)
        try:
            with Image.open(arquivo) as img:
                img.load()
                largura, altura = img.size
                modo = img.mode
        except Exception:
            nao_abriram.append(arquivo)
            continue
        with open(arquivo, "rb") as f:
            impressao = hashlib.md5(f.read()).hexdigest()
        info.append({
            "arquivo": arquivo,
            "classe": os.path.basename(pasta),
            "largura": largura,
            "altura": altura,
            "modo": modo,
            "hash": impressao,
        })

print("Imagens que abriram:", len(info))
print("Arquivos que não abriram:", len(nao_abriram), nao_abriram[:10])
print("Pastas (classes):", Counter(d["classe"] for d in info))
print("Modos de cor:", Counter(d["modo"] for d in info))

* Para observar alguns erros de classificação ou se existem arquivos idênticos (e retirá-las posteriormente)

In [ ]:
por_hash = defaultdict(list)
for d in info:
    por_hash[d["hash"]].append(d)

grupos_identicos = [g for g in por_hash.values() if len(g) > 1]
em_classes_diferentes = [g for g in grupos_identicos if len({d["classe"] for d in g}) > 1]

print("Grupos de arquivos idênticos:", len(grupos_identicos))
print("Desses, em classes diferentes:", len(em_classes_diferentes))
for g in em_classes_diferentes[:10]:
    print([d["arquivo"] for d in g])

In [ ]:
# Grupos com classes diferentes saem inteiros; nos outros, fica um arquivo só.
imagens = [g[0] for g in por_hash.values() if len({d["classe"] for d in g}) == 1]

def impressao_visual(arquivo):
    # Versão 9 x 8 em cinza: cada pixel é comparado com o vizinho da direita.
    with Image.open(arquivo) as im:
        cinza = np.asarray(im.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (cinza[:, 1:] > cinza[:, :-1]).tobytes()

# Imagens quase idênticas ficam com a mesma origem e vão para o mesmo lado na divisão.
origens = {}
for d in imagens:
    d["origem"] = origens.setdefault(impressao_visual(d["arquivo"]), len(origens))

print("Imagens depois de tirar as repetidas:", len(imagens))
print("Origens diferentes:", len(origens))

* Padronizar tamanho das imagens: 224 x 224, mantendo a proporção (lado menor vira 224 e o excesso do lado maior é cortado no centro)

In [ ]:
# Mesmo filtro do notebook original: imagens com algum lado menor que 94 px são descartadas.
LADO_MINIMO = 94
imagens = [d for d in imagens if min(d["largura"], d["altura"]) >= LADO_MINIMO]
print("Imagens com pelo menos", LADO_MINIMO, "px em cada lado:", len(imagens))

LARGURA, ALTURA = 224, 224
ampliadas = sum(1 for d in imagens if min(d["largura"], d["altura"]) < 224)
print("Tamanho-alvo:", LARGURA, "x", ALTURA, "(largura x altura)")
print("Imagens que vão ser ampliadas (lado menor < 224 px):", ampliadas)

In [ ]:
def valor_na_posicao(img, ys, xs):
    # img: (altura, largura, 3). ys, xs: posições na imagem original, que podem ser quebradas.
    # Devolve um array (quantidade de posições, 3) com o valor de cada canal.
    h, w = img.shape[:2]
    y0 = np.clip(np.floor(ys).astype(int), 0, h - 1)
    y1 = np.clip(np.ceil(ys).astype(int), 0, h - 1)
    x0 = np.clip(np.floor(xs).astype(int), 0, w - 1)
    x1 = np.clip(np.ceil(xs).astype(int), 0, w - 1)
    # O maior valor entre os 4 pixels em volta da posição, canal por canal
    return np.max([img[y0, x0], img[y0, x1], img[y1, x0], img[y1, x1]], axis=0)

In [ ]:
def redimensionar(img):
    # O lado menor da imagem vira 224 e o lado maior é cortado no centro. A proporção se mantém.
    h, w = img.shape[:2]
    escala = max(ALTURA / h, LARGURA / w)
    # Posição, na imagem original, do centro de cada pixel da imagem nova
    ys = (h - ALTURA / escala) / 2 + (np.arange(ALTURA) + 0.5) / escala - 0.5
    xs = (w - LARGURA / escala) / 2 + (np.arange(LARGURA) + 0.5) / escala - 0.5
    yy, xx = np.meshgrid(ys, xs, indexing="ij")
    return valor_na_posicao(img, yy.ravel(), xx.ravel()).reshape(ALTURA, LARGURA, 3)

## 3. Divisão entre Treino, Validação e Teste

* A divisão é feita **antes** do data augmentation, só com as imagens originais. Assim, nada que vem de uma imagem de validação ou de teste pode aparecer no treino.
* Imagens quase idênticas (mesma `origem`) ficam sempre do mesmo lado.

In [ ]:
grupo = np.array([d["origem"] for d in imagens])

def separar_por_grupo(indices, fracao):
    # Sorteia grupos inteiros até juntar a fração pedida das imagens.
    contagem = Counter(grupo[indices].tolist())
    alvo = round(fracao * len(indices))
    escolhidos, total = [], 0
    for g in rng.permutation(sorted(contagem)):
        if total >= alvo:
            break
        escolhidos.append(g)
        total += contagem[g]
    separado = np.isin(grupo[indices], escolhidos)
    return indices[~separado], indices[separado]

# Posições na lista `imagens` (ainda sem nenhuma imagem aumentada)
treino_original, teste = separar_por_grupo(np.arange(len(imagens)), 0.15)
treino_original, validacao = separar_por_grupo(treino_original, 0.15)

print("Treino:", len(treino_original), "| validação:", len(validacao), "| teste:", len(teste))

## 4. Medidas de Regularização

* O data augmentation é aplicado **somente nas imagens de treino**.

#### Data Augmentation

-> Rotação 15° no sentido horário

In [ ]:
def girar_horario(img, graus=15):
    h, w = img.shape[:2]
    cy, cx = (h - 1) / 2, (w - 1) / 2
    t = np.deg2rad(graus)

    # Para cada pixel da imagem girada, de onde ele vem na original
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    dy, dx = yy - cy, xx - cx
    xs = dx * np.cos(t) + dy * np.sin(t) + cx
    ys = -dx * np.sin(t) + dy * np.cos(t) + cy
    dentro = (xs >= 0) & (xs <= w - 1) & (ys >= 0) & (ys <= h - 1)

    nova = np.zeros((h, w, 3), dtype=np.float32)
    nova[dentro] = valor_na_posicao(img, ys[dentro], xs[dentro])

    # Cantos vazios: média dos 3 pixels válidos mais próximos, canal por canal
    validos = np.argwhere(dentro)
    vazios = np.argwhere(~dentro)
    if len(vazios) > 0:
        _, vizinhos = cKDTree(validos).query(vazios, k=3)
        pos = validos[vizinhos]                      # (vazios, 3 vizinhos, 2 coordenadas)
        nova[vazios[:, 0], vazios[:, 1]] = nova[pos[..., 0], pos[..., 1]].mean(axis=1)
    return nova

#### Adicionar Ruído

In [ ]:
def adicionar_ruido(img):
    ruidosa = img + rng.normal(0, 2, size=img.shape)
    return np.clip(ruidosa, 0, 255).astype(np.float32)

#### Aplicação em alguns dados do treino selecionados aleatoriamente

In [ ]:
n_treino = len(treino_original)
ordem = rng.permutation(treino_original)
n_rotacao = round(0.10 * n_treino)
n_ruido = round(0.10 * n_treino)
vao_girar = ordem[:n_rotacao].tolist()
vao_ruido = ordem[n_rotacao:n_rotacao + n_ruido].tolist()
print("Vão ganhar uma versão girada:", len(vao_girar), "| com ruído:", len(vao_ruido), "| (de", n_treino, "imagens de treino)")

#### Carregamento das imagens

* As imagens ficam guardadas como `uint8` (0 a 255). A conversão para `float` (0 a 1) é feita por lote, na hora de treinar.
* As linhas `0 .. n-1` são as imagens originais (mesma posição da lista `imagens`). As versões novas (giradas ou com ruído) ficam depois delas e só entram no treino.

In [ ]:
classes = sorted({d["classe"] for d in imagens})
n = len(imagens)
total = n + len(vao_girar) + len(vao_ruido)

X = np.empty((total, ALTURA, LARGURA, 3), dtype=np.uint8)   # (n, altura, largura, 3), de 0 a 255
y = np.empty(total, dtype=np.int64)

# Linha onde cada versão nova vai ficar
linha_nova = {}
for k, i in enumerate(vao_girar):
    linha_nova[i] = n + k
for k, i in enumerate(vao_ruido):
    linha_nova[i] = n + len(vao_girar) + k
girar, ruido = set(vao_girar), set(vao_ruido)

def para_uint8(v):
    return np.clip(np.rint(v), 0, 255).astype(np.uint8)

for i, d in enumerate(imagens):
    with Image.open(d["arquivo"]) as im:
        img = np.asarray(im.convert("RGB"), dtype=np.float32)   # valores de 0 a 255

    base = redimensionar(img)
    X[i] = para_uint8(base)
    y[i] = classes.index(d["classe"])

    # A rotação é feita antes de redimensionar; o ruído, depois.
    if i in girar:
        X[linha_nova[i]] = para_uint8(redimensionar(girar_horario(img)))
        y[linha_nova[i]] = y[i]
    elif i in ruido:
        X[linha_nova[i]] = para_uint8(adicionar_ruido(base))
        y[linha_nova[i]] = y[i]

X = torch.from_numpy(X)
y = torch.from_numpy(y)
dados = TensorDataset(X, y)

# Treino = originais do treino + versões novas. Validação e teste = só originais.
treino = np.concatenate([treino_original, np.arange(n, total)])

print("Classes:", classes)
print("Total de imagens (originais + novas):", len(y), "| formato:", tuple(X.shape), "| tipo:", X.dtype)
print("Treino:", len(treino), "| validação:", len(validacao), "| teste:", len(teste))

## 5. Modelo

* Obs.: A quantidade de camadas convolucionais e de camadas totalmente conectadas foram inspiradas no modelo AlexNet, apresentado no livro Understanding Deep Learning - capítulo 10.

* As outras decisões foram escolhidas e documentadas em ``historico_decisoes.ipynb``, onde o assistente Claude IA foi responsável pela realização de ensaios com o número reduzido de dados para testar as possíveis variações nos parâmetros e assim podermos selecionar a combinação que melhorava o aprendizado do modelo.
* Esses ensaios foram feitos com imagens 100 x 94. Aqui os parâmetros foram mantidos iguais; só o tamanho da entrada mudou (a rede calcula sozinha o tamanho da camada totalmente conectada: com 224 x 224 sobram 5 x 5 pixels por filtro).

In [ ]:
FILTROS = 32

class Rede(nn.Module):
    def __init__(self, altura, largura):
        super().__init__()

        # 5 blocos: convolução -> ReLU -> max pooling -> dropout
        blocos = []
        canais_entrada = 3
        for _ in range(5):
            blocos += [
                nn.Conv2d(canais_entrada, FILTROS, kernel_size=5, stride=1, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2),
                nn.Dropout(0.2),
            ]
            canais_entrada = FILTROS
        self.convolucoes = nn.Sequential(*blocos)

        # Quantos pixels sobram depois dos 5 blocos
        for _ in range(5):
            altura = (altura - 2) // 2
            largura = (largura - 2) // 2

        # Flatten -> 1024 -> ReLU -> 32 -> ReLU -> 2
        self.conectadas = nn.Sequential(
            nn.Flatten(),
            nn.Linear(FILTROS * altura * largura, 1024), nn.ReLU(),
            nn.Linear(1024, 32), nn.ReLU(),
            nn.Linear(32, 2),
        )

    def antes_da_softmax(self, x):
        return self.conectadas(self.convolucoes(x))

    def forward(self, x):
        return torch.softmax(self.antes_da_softmax(x), dim=1)

## 6. Treino e Validação dos Dados

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo = Rede(ALTURA, LARGURA).to(dispositivo)
otimizador = torch.optim.SGD(modelo.parameters(), lr=1e-2)
lotes_treino = DataLoader(Subset(dados, treino), batch_size=32, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))

def preparar(xb):
    # uint8 (lote, altura, largura, 3) -> float (lote, 3, altura, largura), de 0 a 1
    return xb.to(dispositivo).permute(0, 3, 1, 2).float().div(255).contiguous()

def avaliar(indices):
    # Devolve (perda, acurácia) nas imagens pedidas, com dropout desligado.
    modelo.eval()
    acertos, soma_perda = 0, 0.0
    with torch.no_grad():
        for xb, yb in DataLoader(Subset(dados, indices), batch_size=32):
            z = modelo.antes_da_softmax(preparar(xb))
            yb = yb.to(dispositivo)
            soma_perda += F.cross_entropy(z, yb, reduction="sum").item()
            acertos += (z.argmax(dim=1) == yb).sum().item()
    modelo.train()
    return soma_perda / len(indices), acertos / len(indices)

def acuracia(indices):
    return avaliar(indices)[1]

In [ ]:
EPOCAS = 30   # o notebook original usava 100

historico = {"perda_treino": [], "acc_treino": [], "perda_val": [], "acc_val": []}
melhor_acuracia = 0

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    soma_perda, acertos, vistos = 0.0, 0, 0
    for xb, yb in lotes_treino:
        xb, yb = preparar(xb), yb.to(dispositivo)
        z = modelo.antes_da_softmax(xb)
        perda = F.nll_loss(F.log_softmax(z, dim=1), yb)   # entropia cruzada da saída da softmax
        otimizador.zero_grad()
        perda.backward()
        otimizador.step()
        # Perda e acurácia de treino: média durante a época, com dropout ligado.
        soma_perda += perda.item() * len(yb)
        acertos += (z.argmax(dim=1) == yb).sum().item()
        vistos += len(yb)

    perda_val, acc = avaliar(validacao)
    historico["perda_treino"].append(soma_perda / vistos)
    historico["acc_treino"].append(acertos / vistos)
    historico["perda_val"].append(perda_val)
    historico["acc_val"].append(acc)
    json.dump(historico, open("historico_treino_224.json", "w"))
    print(f"Época {epoca:3d}   perda no treino: {soma_perda / vistos:.4f}   acurácia no treino: {acertos / vistos:.3f}   "
          f"acurácia na validação: {acc:.3f}")

    if acc > melhor_acuracia:
        melhor_acuracia, melhor_epoca = acc, epoca
        melhores_pesos = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
        torch.save(melhores_pesos, "modelo_cats_and_dogs_224.pt")

# O modelo final é o da época com a melhor acurácia na validação.
modelo.load_state_dict(melhores_pesos)
print(f"Melhor época: {melhor_epoca} | acurácia na validação: {melhor_acuracia:.3f}")

#### Observação sobre o treino: overfitting

* Durante o treino, a perda no treino caiu para praticamente zero, enquanto a perda na validação ficou alta o tempo todo e a acurácia na validação ficou oscilando. Isso indica **overfitting**: o modelo decorou as imagens de treino e não generalizou bem para imagens novas.
* Com a entrada em 224 x 224 há mais detalhes para decorar e a primeira camada totalmente conectada fica bem maior (800 entradas, contra ~64 no notebook de 100 x 94), enquanto a regularização continuou a mesma.
* **Próximo passo sugerido:** aumentar o data augmentation, aplicando transformações aleatórias a cada época em todas as imagens de treino (por exemplo, espelhamento horizontal e recortes aleatórios), em vez de só 20% delas ganharem uma cópia fixa. Outras opções: dropout nas camadas totalmente conectadas e weight decay.
* Como a validação já foi usada para escolher a melhor época, o conjunto de **teste só é aberto agora, no final**, e não foi usado para nenhuma decisão.

## 7. Avaliação final no Teste

In [ ]:
print("Acurácia no teste:", acuracia(teste))

## 8. Análise dos Resultados

#### Gráficos Acurácia e Perda no Treino

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Usa o modelo fechado, salvo na célula de treino. Nada aqui treina de novo.
modelo.load_state_dict(torch.load("modelo_cats_and_dogs_224.pt", map_location=dispositivo))
modelo.eval()

AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#8a8985"
TINTA, TINTA_2, GRADE = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "text.color": TINTA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.edgecolor": GRADE, "axes.grid": True, "grid.color": GRADE, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

# Perda e acurácia de treino: média durante a época, com dropout ligado.
hist = json.load(open("historico_treino_224.json"))
epocas = np.arange(1, len(hist["acc_val"]) + 1)
ep = int(np.argmax(hist["acc_val"])) + 1   # melhor época na validação (a primeira, se houver empate)

fig, (ax_acc, ax_perda) = plt.subplots(1, 2, figsize=(13, 4.8))

ax_acc.axhline(0.5, color=TINTA_2, lw=1, ls=":")
ax_acc.text(epocas[-1], 0.51, "acaso (50%)", color=TINTA_2, fontsize=9, ha="right")
ax_acc.plot(epocas, hist["acc_treino"], color=AZUL, lw=2, label="Treino")
ax_acc.plot(epocas, hist["acc_val"], color=LARANJA, lw=2, label="Validação")
ax_acc.plot(ep, hist["acc_val"][ep - 1], "o", ms=9, color=TINTA, mec="#fcfcfb", mew=2, zorder=5)
ax_acc.annotate(f"Modelo fechado\nÉpoca {ep}: {hist['acc_val'][ep - 1]:.1%}",
                (ep, hist["acc_val"][ep - 1]), xytext=(epocas[-1] * 0.45, 0.62), color=TINTA, fontsize=10,
                arrowprops=dict(arrowstyle="-", color=TINTA_2, lw=1))
ax_acc.set(title="Acurácia por época", xlabel="Época", ylabel="Acurácia", xlim=(1, epocas[-1]), ylim=(0.45, 1.0))
ax_acc.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax_acc.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 0.08), fontsize=9.5)

ax_perda.plot(epocas, hist["perda_treino"], color=AZUL, lw=2, label="Treino")
ax_perda.plot(epocas, hist["perda_val"], color=LARANJA, lw=2, label="Validação")
ax_perda.set(title="Perda (entropia cruzada) por época", xlabel="Época", ylabel="Perda", xlim=(1, epocas[-1]), ylim=(0, None))
ax_perda.legend(frameon=False, loc="upper right", fontsize=9.5)

fig.tight_layout()
plt.show()

#### Matriz de Confusão 

In [ ]:
NOMES = {"Cat": "gato", "Dog": "cachorro"}
rotulos = [NOMES[c] for c in classes]

# Previsões do modelo fechado no conjunto de teste
probs = []
with torch.no_grad():
    for xb, _ in DataLoader(Subset(dados, teste), batch_size=32):
        probs.append(modelo(preparar(xb)).cpu())
probs = torch.cat(probs)
previsto = probs.argmax(dim=1)
real = y[teste]

matriz = np.zeros((2, 2), dtype=int)
for r_, p_ in zip(real.tolist(), previsto.tolist()):
    matriz[r_, p_] += 1

print(f"Acertos: {matriz.trace()} de {matriz.sum()} ({matriz.trace() / matriz.sum():.4f})")
for i, nome in enumerate(rotulos):
    print(f"{nome:9s} recall {matriz[i, i] / matriz[i].sum():.1%}   precisão {matriz[i, i] / matriz[:, i].sum():.1%}")

azuis = LinearSegmentedColormap.from_list("azuis", ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
fig, ax = plt.subplots(figsize=(5.8, 5))
ax.imshow(matriz, cmap=azuis, vmin=0)
for i in range(2):
    for j in range(2):
        cor = "white" if matriz[i, j] > matriz.max() / 2 else TINTA
        ax.text(j, i - 0.07, f"{matriz[i, j]}", ha="center", va="center", color=cor, fontsize=22, fontweight="bold")
        ax.text(j, i + 0.13, f"{matriz[i, j] / matriz[i].sum():.1%} dos {rotulos[i]}s", ha="center", va="center", color=cor, fontsize=10)
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=rotulos, yticklabels=rotulos,
       xlabel="Classe prevista", ylabel="Classe real", title="Matriz de confusão no teste")
ax.grid(False)
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_xticks([0.5], minor=True)
ax.set_yticks([0.5], minor=True)
ax.grid(which="minor", color="#fcfcfb", lw=3)
ax.tick_params(which="minor", length=0)
fig.tight_layout()
plt.show()

#### Exemplos de imagens que foram classificadas errado

In [ ]:
def mostrar_erros(classe_real, quantos=12):
    classe_prevista = 1 - classe_real
    erros = ((real == classe_real) & (previsto == classe_prevista)).nonzero().ravel()
    # Do mais confiante para o menos; uma imagem por origem, para não repetir a mesma foto.
    erros = erros[probs[erros, classe_prevista].argsort(descending=True)]
    vistos, escolhidos = set(), []
    for e in erros.tolist():
        origem = int(grupo[teste[e]])
        if origem not in vistos:
            vistos.add(origem)
            escolhidos.append(e)
        if len(escolhidos) == quantos:
            break

    colunas = 6
    linhas = int(np.ceil(len(escolhidos) / colunas))
    fig, eixos = plt.subplots(linhas, colunas, figsize=(13, 2.5 * linhas), squeeze=False)
    for ax in eixos.ravel():
        ax.axis("off")
    for ax, e in zip(eixos.ravel(), escolhidos):
        ax.imshow(X[teste[e]].numpy(), interpolation="nearest")
        ax.set_title(f"{probs[e, classe_prevista]:.1%} de certeza", fontsize=10, fontweight="normal", color=TINTA_2, loc="center")
    fig.suptitle(f"Era {rotulos[classe_real]}, previu {rotulos[classe_prevista]}: "
                 f"{len(erros)} erros no teste. Estes são os {len(escolhidos)} com maior certeza.",
                 x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

mostrar_erros(classes.index("Cat"))
mostrar_erros(classes.index("Dog"))